# Evaluation

This notebook evaluates our job-matching RAG assistant, the same way we did in
the course: first for the FAQ assistant, now adapted to our own dataset of
data-analyst job listings for Israel.

We answer three questions:

1. **Search evaluation** — when someone searches with a query, does the
   right job actually come back in the results?
2. **RAG answer evaluation** — when the assistant writes a recommendation
   in plain English, is that recommendation actually correct?
3. **Cost** — how much does it cost (in OpenAI API usage) to run all of this?

The course notebook worked with a static FAQ (question → fixed answer). Our
project is different: we don't have a fixed "correct answer" text sitting in
the data — we have structured job listings (title, company, skills, etc.),
and the assistant *writes* a recommendation instead of returning a stored
answer. We adapt the same evaluation logic to that difference, and call it
out wherever it matters.

**Note on running this notebook:** every LLM call here costs a small amount
of money (a fraction of a cent, but it adds up). To keep costs low we only
generate ground truth from a handful of job listings, not the full 4,000+.


# 1. Generating Ground Truth Data

"Ground truth" means: for a bunch of realistic search queries, we know in
advance which job listing is the *correct* one to find. We need this before
we can measure whether search (or the RAG assistant) is doing a good job —
otherwise we have no way to check if the results are actually right.

We'll generate ground truth automatically: for each job listing, we ask an
LLM to imagine 5 realistic queries a job seeker might type that this
specific listing would be a good answer to.


In the course, `ingest.py` lived in a separate folder per module, so the
notebook had to add that folder to `sys.path` before importing it. In this
project `ingest.py` already lives at the project root, right next to this
notebook, so we can import it directly — no path setup needed.

We also don't need to hit BigQuery again here. We already loaded every job
listing once and saved it into `jobs.db` when we ran `ingest.py`. So instead
of `load_jobs_data()` (which queries BigQuery), we use
`load_documents_from_db()`, which just reads the documents straight back out
of `jobs.db`. Faster, free, and gives us the exact same data we searched
over.


In [8]:
from ingest import load_jobs_data

documents = load_jobs_data()

Loaded 1 job listings
Loaded 2 job listings
Loaded 3 job listings
Loaded 4 job listings
Loaded 5 job listings
Loaded 6 job listings
Loaded 7 job listings
Loaded 8 job listings
Loaded 9 job listings
Loaded 10 job listings
Loaded 11 job listings
Loaded 12 job listings
Loaded 13 job listings
Loaded 14 job listings
Loaded 15 job listings
Loaded 16 job listings
Loaded 17 job listings
Loaded 18 job listings
Loaded 19 job listings
Loaded 20 job listings
Loaded 21 job listings
Loaded 22 job listings
Loaded 23 job listings
Loaded 24 job listings
Loaded 25 job listings
Loaded 26 job listings
Loaded 27 job listings
Loaded 28 job listings
Loaded 29 job listings
Loaded 30 job listings
Loaded 31 job listings
Loaded 32 job listings
Loaded 33 job listings
Loaded 34 job listings
Loaded 35 job listings
Loaded 36 job listings
Loaded 37 job listings
Loaded 38 job listings
Loaded 39 job listings
Loaded 40 job listings
Loaded 41 job listings
Loaded 42 job listings
Loaded 43 job listings
Loaded 44 job listin

In the course, the FAQ dataset covered several courses, so the next step was
filtering down to just `llm-zoomcamp`. We don't need that here — our dataset
is already just one thing: data-analyst job listings for Israel. Every
document in `documents` is already in scope.


In [9]:
# How many job listings did we load?
len(documents)


4255

In [10]:
# Each document is a dictionary with fields like Title, Company_Name, City,
# Job_Description, skills, experience_bucket, and Link.
# Link acts as our unique id (there's no separate "id" field).
documents[0]

{'Title': 'Technical Support Engineer III, Israel',
 'Job_Description': "Cohesity is the leader in AI-powered data security. Over 13,600 enterprise customers, including over 85 of the Fortune 100 and nearly 70% of the Global 500, rely on Cohesity to strengthen their resilience while providing Gen AI insights into their vast amounts of data. Formed from the combination of Cohesity with Veritas’ enterprise data protection business, the company’s solutions secure and protect data on-premises, in the cloud, and at the edge. Backed by NVIDIA, IBM, HPE, Cisco, AWS, Google Cloud, and others, Cohesity is headquartered in Santa Clara, CA, with offices around the globe.We’ve been named a Leader by multiple analyst firms and have been globally recognized for Innovation, Product Strength, and Simplicity in Design , and our culture.Want to join the leader in AI-powered data security?Join Cohesity as a Technical Support Engineer and become part of a team that is reshaping data management.This role w

In [11]:
# Take one job listing to experiment with
doc = documents[0]
print(doc["Link"])
print(doc["Title"])
print(doc["Job_Description"])

https://il.linkedin.com/jobs/view/technical-support-engineer-iii-israel-at-cohesity-4415367466
Technical Support Engineer III, Israel
Cohesity is the leader in AI-powered data security. Over 13,600 enterprise customers, including over 85 of the Fortune 100 and nearly 70% of the Global 500, rely on Cohesity to strengthen their resilience while providing Gen AI insights into their vast amounts of data. Formed from the combination of Cohesity with Veritas’ enterprise data protection business, the company’s solutions secure and protect data on-premises, in the cloud, and at the edge. Backed by NVIDIA, IBM, HPE, Cisco, AWS, Google Cloud, and others, Cohesity is headquartered in Santa Clara, CA, with offices around the globe.We’ve been named a Leader by multiple analyst firms and have been globally recognized for Innovation, Product Strength, and Simplicity in Design , and our culture.Want to join the leader in AI-powered data security?Join Cohesity as a Technical Support Engineer and become

## Generating questions with structured output

With structured output, we ask the LLM to return data in a specific format
instead of free-form text. For example, instead of getting a paragraph that
contains questions, we can ask for a Python object with a `questions` field.


In [12]:
# We want the output as a list of strings, so we define that structure with a Pydantic model

from pydantic import BaseModel

class Questions(BaseModel):
    questions: list[str]

In [13]:
data_gen_instructions = """
You emulate a person job hunting for data analyst roles in Israel.
Formulate 5 search queries this job seeker might type into a job search assistant,
based on this job listing. The listing contains the information needed to answer
these queries, such as the title, skills, experience level, city, and description.
If possible, use as few exact words from the listing as possible.

The output should resemble how people search for jobs online. Not too formal,
not too short, not too long.
""".strip()

In [14]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

In [15]:
import json

user_prompt = json.dumps(doc)

In [16]:
messages = [
    {"role": "developer", "content": data_gen_instructions},
    {"role": "user", "content": user_prompt}
]

Until now we called `responses.create` and read `response.output_text`. For
structured output we switch to `responses.parse` and pass
`text_format=Questions`, which tells the API to return our class instead of
free text.


In [17]:
response = openai_client.responses.parse(
    model="gpt-5.4-mini",
    input=messages,
    text_format=Questions
)

In [18]:
result = response.output_parsed

print(result)

questions=['support engineer jobs in Herzliya with storage or backup systems', 'technical support role in Israel for cloud and virtualization', 'enterprise support engineer position near Tel Aviv for networking and OS troubleshooting', 'job opening in Israel for data protection / backup appliances support', 'customer-facing tech support job in Herzliya with night or weekend shifts']


In [19]:
#genereted questions using the doc variable
print(result.questions)

['support engineer jobs in Herzliya with storage or backup systems', 'technical support role in Israel for cloud and virtualization', 'enterprise support engineer position near Tel Aviv for networking and OS troubleshooting', 'job opening in Israel for data protection / backup appliances support', 'customer-facing tech support job in Herzliya with night or weekend shifts']


## Reusable utilities
We are doign the same we did before but now using the evaluation_utils.py file, where we have the fuctions ready to use


In [20]:
# Import the structured-output helper:
from evaluation_utils import llm_structured, calc_total_price

In [21]:
# check if we get the same questions from the same doc variable using the llm_structured function
result, usage = llm_structured(
    openai_client,
    data_gen_instructions,
    user_prompt,
    Questions
)

print(result.questions)

['data support engineer jobs Herzliya', 'enterprise tech support role in Tel Aviv area storage and cloud', 'job for troubleshooting backups and virtualization Israel', 'technical support position with Linux networking and storage skills Herzliya', 'customer support engineer for data protection software in Israel']


In [22]:
# calculate tokens used
usage.input_tokens, usage.output_tokens

(1117, 62)

In [23]:
from evaluation_utils import calc_price

In [24]:
# calculate cost of the API call based on the token usage
cost = calc_price(usage)
cost

{'input_cost': 0.00083775, 'output_cost': 0.000279, 'total_cost': 0.00111675}

In [25]:
# list of dictionaries with the generated questions and the corresponding document link
records = []

for q in result.questions:
    records.append({
        "question": q,
        "document": doc["Link"]
    })

records

[{'question': 'data support engineer jobs Herzliya',
  'document': 'https://il.linkedin.com/jobs/view/technical-support-engineer-iii-israel-at-cohesity-4415367466'},
 {'question': 'enterprise tech support role in Tel Aviv area storage and cloud',
  'document': 'https://il.linkedin.com/jobs/view/technical-support-engineer-iii-israel-at-cohesity-4415367466'},
 {'question': 'job for troubleshooting backups and virtualization Israel',
  'document': 'https://il.linkedin.com/jobs/view/technical-support-engineer-iii-israel-at-cohesity-4415367466'},
 {'question': 'technical support position with Linux networking and storage skills Herzliya',
  'document': 'https://il.linkedin.com/jobs/view/technical-support-engineer-iii-israel-at-cohesity-4415367466'},
 {'question': 'customer support engineer for data protection software in Israel',
  'document': 'https://il.linkedin.com/jobs/view/technical-support-engineer-iii-israel-at-cohesity-4415367466'}]

# 2. Generating Ground Truth for a Sample of Job Listings

We have ~4,000+ job listings. Generating 5 questions per listing for all of
them would mean thousands of LLM calls — slow and unnecessarily expensive
for a demo evaluation. Just like in the course, we'll generate ground truth
from a small sample (the first 5 listings), which already gives us 25
question/answer pairs to evaluate with. You can raise this number later if
you want a bigger evaluation set.


In [26]:
def generate_ground_truth(doc):
    user_prompt = json.dumps(doc)

    out, usage = llm_structured_retry(
        openai_client,
        data_gen_instructions,
        user_prompt,
        Questions
    )

    results = []

    for q in out.questions:
        results.append({
            "question": q,
            "document": doc["Link"]
        })

    return results, usage

In [27]:
from tqdm.auto import tqdm
from evaluation_utils import llm_structured_retry

ground_truth = []
usages = []

for doc in tqdm(documents[:5]):
    records, usage = generate_ground_truth(doc)
    ground_truth.extend(records)
    usages.append(usage)

  0%|          | 0/5 [00:00<?, ?it/s]

In [28]:
ground_truth

[{'question': 'support engineer jobs in Herzliya Israel with storage or backup systems',
  'document': 'https://il.linkedin.com/jobs/view/technical-support-engineer-iii-israel-at-cohesity-4415367466'},
 {'question': 'technical support role in Tel Aviv area working on cloud and databases',
  'document': 'https://il.linkedin.com/jobs/view/technical-support-engineer-iii-israel-at-cohesity-4415367466'},
 {'question': 'enterprise support engineer opening Israel troubleshooting Linux and networking tools',
  'document': 'https://il.linkedin.com/jobs/view/technical-support-engineer-iii-israel-at-cohesity-4415367466'},
 {'question': 'job for data protection / backup support engineer near Herzliya',
  'document': 'https://il.linkedin.com/jobs/view/technical-support-engineer-iii-israel-at-cohesity-4415367466'},
 {'question': 'customer-facing tech support position in Israel with nights or shift work',
  'document': 'https://il.linkedin.com/jobs/view/technical-support-engineer-iii-israel-at-cohesi

In [29]:
usages

[ResponseUsage(input_tokens=1117, input_tokens_details=InputTokensDetails(cache_write_tokens=0, cached_tokens=0), output_tokens=75, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=1192),
 ResponseUsage(input_tokens=767, input_tokens_details=InputTokensDetails(cache_write_tokens=0, cached_tokens=0), output_tokens=62, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=829),
 ResponseUsage(input_tokens=732, input_tokens_details=InputTokensDetails(cache_write_tokens=0, cached_tokens=0), output_tokens=57, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=789),
 ResponseUsage(input_tokens=807, input_tokens_details=InputTokensDetails(cache_write_tokens=0, cached_tokens=0), output_tokens=66, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=873),
 ResponseUsage(input_tokens=740, input_tokens_details=InputTokensDetails(cache_write_tokens=0, cached_tokens=0), output_tokens=58, output_tokens_d

In [30]:
import pandas as pd

df_ground_truth = pd.DataFrame(ground_truth)

In [31]:
df_ground_truth.to_csv("data/ground_truth.csv", index=False)

## Search Evaluation


In the course this next part was a separate notebook, so it started by
reloading the ground truth CSV from disk. We keep everything in one
notebook, but it's still useful to see how to reload it — that way you can
re-run just the evaluation later without spending money generating new
ground truth.


In [32]:
# Load the ground truth we generated above (or saved earlier)
# ground truth equals the questions we generated for each document, and the corresponding document link
df_ground_truth = pd.read_csv("data/ground_truth.csv")
df_ground_truth.head(2)

,question,document
0,support engineer jobs in Herzliya Israel with ...,https://il.linkedin.com/jobs/view/technical-su...
1,technical support role in Tel Aviv area workin...,https://il.linkedin.com/jobs/view/technical-su...


In [33]:
# is a pandas DataFrame method that converts a table into a list of dictionaries, where each row becomes one dictionary
ground_truth = df_ground_truth.to_dict(orient="records")
ground_truth[0]

{'question': 'support engineer jobs in Herzliya Israel with storage or backup systems',
 'document': 'https://il.linkedin.com/jobs/view/technical-support-engineer-iii-israel-at-cohesity-4415367466'}

In [34]:
# remember we already have:
doc = documents[0]
print(doc["Link"])
print(doc["Title"])
print(doc["Job_Description"])

https://il.linkedin.com/jobs/view/technical-support-engineer-iii-israel-at-cohesity-4415367466
Technical Support Engineer III, Israel
Cohesity is the leader in AI-powered data security. Over 13,600 enterprise customers, including over 85 of the Fortune 100 and nearly 70% of the Global 500, rely on Cohesity to strengthen their resilience while providing Gen AI insights into their vast amounts of data. Formed from the combination of Cohesity with Veritas’ enterprise data protection business, the company’s solutions secure and protect data on-premises, in the cloud, and at the edge. Backed by NVIDIA, IBM, HPE, Cisco, AWS, Google Cloud, and others, Cohesity is headquartered in Santa Clara, CA, with offices around the globe.We’ve been named a Leader by multiple analyst firms and have been globally recognized for Innovation, Product Strength, and Simplicity in Design , and our culture.Want to join the leader in AI-powered data security?Join Cohesity as a Technical Support Engineer and become

We already built and saved a keyword search index in `ingest.py`
(`jobs.db`). We don't need to rebuild it here — we just load it, the same
way `rag_helper.py` does.


In [35]:
import rag_helper

# search index = the organized memory of your documents that
# lets RAG find relevant context before asking the LLM
index = rag_helper.load_index()

In [36]:
# boost_dict means: give more importance to some fields when searching.
# Title is the most specific identifying text (like "question" in the FAQ),
# skills is more of a supporting/categorical signal (like "section").
# return index.search(...) searches the index for documents relevant to the query and returns the top matching results.
def text_search(query):
    boost_dict = {"Title": 3.0, "skills": 0.5}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict
    )

## Collecting relevance data


In [37]:
ground_truth[0]

{'question': 'support engineer jobs in Herzliya Israel with storage or backup systems',
 'document': 'https://il.linkedin.com/jobs/view/technical-support-engineer-iii-israel-at-cohesity-4415367466'}

In [38]:
# Run search for this question
rec = ground_truth[0]
doc_id = rec["document"]
results = text_search(query=rec["question"])
results

[{'Title': 'Pharmacovigilance Associate & Country Safety Head Backup',
  'Job_Description': "Location: Yakum, IsraelHiring Manager: Irena Shinkar8-month fixed-term positionAbout The JobAs a Pharmacovigilance Associate and Country Safety Head Backup, you will support pharmacovigilance activities in Israel and serve as backup to the Country Safety Head and QPPV, ensuring continuous compliance with local and international pharmacovigilance regulations while maintaining the highest standards of patient safety.Your key responsibilities will include monitoring diverse safety information sources to ensure timely detection and appropriate management of incoming pharmacovigilance data, assisting in the overall evaluation of pharmacovigilance information, and reporting safety related information to relevant stakeholders including Health Authorities. You will support local safety surveillance activities, including the management of safety signals and assessment of new safety information, and supp

In [39]:
for d in results:
    print(f'{d["Link"]} == {doc_id}: {d["Link"] == doc_id}')

https://il.linkedin.com/jobs/view/pharmacovigilance-associate-country-safety-head-backup-at-sanofi-4433082645 == https://il.linkedin.com/jobs/view/technical-support-engineer-iii-israel-at-cohesity-4415367466: False
https://il.linkedin.com/jobs/view/technical-support-engineer-iii-israel-at-cohesity-4415367466 == https://il.linkedin.com/jobs/view/technical-support-engineer-iii-israel-at-cohesity-4415367466: True
https://www.indeed.com/viewjob?jk=4e4c9d4656b02bc8 == https://il.linkedin.com/jobs/view/technical-support-engineer-iii-israel-at-cohesity-4415367466: False
https://www.indeed.com/viewjob?jk=1dfdc65dd7d56e8c == https://il.linkedin.com/jobs/view/technical-support-engineer-iii-israel-at-cohesity-4415367466: False
https://www.indeed.com/viewjob?jk=cdb4f6cbaab8dae5 == https://il.linkedin.com/jobs/view/technical-support-engineer-iii-israel-at-cohesity-4415367466: False


Then turn this comparison into a relevance list. In this lesson, relevance
means whether a retrieved document is the correct document for this
question.


In [40]:
relevance = []

for d in results:
    relevance.append(int(d["Link"] == doc_id))

relevance

[0, 1, 0, 0, 0]

In [41]:
def compute_relevance_text(rec):
    doc_id = rec["document"]
    results = text_search(query=rec["question"])

    relevance = []
    for d in results:
        relevance.append(int(d["Link"] == doc_id))

    return relevance

In [42]:
rec = ground_truth[0]
print(rec["question"])
compute_relevance_text(rec)
# [1, 0, 0, 0, 0]

support engineer jobs in Herzliya Israel with storage or backup systems


[0, 1, 0, 0, 0]

In [43]:
def compute_relevance_total_text(ground_truth):
    relevance_total = []

    for rec in tqdm(ground_truth):
        relevance = compute_relevance_text(rec)
        relevance_total.append(relevance)

    return relevance_total

In [44]:
ground_truth_sample = ground_truth[:15]
relevance_total_text = compute_relevance_total_text(ground_truth_sample)

  0%|          | 0/15 [00:00<?, ?it/s]

In [45]:
relevance_total_text

[[0, 1, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 1, 0, 0],
 [0, 0, 1, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 1],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 1, 0, 0, 0]]

Next, make the relevance functions generic. We start with text (keyword)
search, but later we may want to evaluate vector search, hybrid search, or
another retrieval method. The relevance logic is the same. Only the search
function changes.


In [46]:
def compute_relevance(rec, search_function):
    doc_id = rec["document"]
    results = search_function(query=rec["question"])

    relevance = []
    for d in results:
        relevance.append(int(d["Link"] == doc_id))

    return relevance

The total relevance function gets a `search_function` too.


In [47]:
def compute_relevance_total(ground_truth, search_function):
    relevance_total = []

    for rec in tqdm(ground_truth):
        relevance = compute_relevance(rec, search_function)
        relevance_total.append(relevance)

    return relevance_total

In [48]:
relevance_total = compute_relevance_total(ground_truth_sample, text_search)
relevance_total

  0%|          | 0/15 [00:00<?, ?it/s]

[[0, 1, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 1, 0, 0],
 [0, 0, 1, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 1],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 1, 0, 0, 0]]

In [49]:
relevance_total = compute_relevance_total(ground_truth, text_search)

  0%|          | 0/25 [00:00<?, ?it/s]

In [50]:
relevance_total

[[0, 1, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 1, 0, 0],
 [0, 0, 1, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 1],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 1, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 1, 0, 0],
 [0, 0, 0, 0, 1],
 [0, 0, 0, 1, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 0, 0, 0],
 [0, 0, 1, 0, 0],
 [0, 0, 0, 0, 0]]

# 3. Search Evaluation Metrics

In the previous part, we computed relevance lists for search results. We
can turn those lists into metrics.


Hit Rate (also called Recall@k) measures the fraction of queries where the
correct document appears anywhere in the results:


## Hit Rate


In [51]:
def hit_rate(relevance):
    cnt = 0

    for line in relevance:
        if 1 in line:
            cnt = cnt + 1

    return cnt / len(relevance)

In [52]:
# let's check it with an example

example = [
    [1, 0, 0, 0, 0],
    [0, 1, 0, 0, 0],
    [1, 0, 0, 0, 0],
    [0, 0, 0, 0, 0],
    [0, 1, 0, 0, 0],
    [1, 0, 0, 0, 0],
    [1, 0, 0, 0, 0],
    [1, 0, 0, 0, 0],
    [1, 0, 0, 0, 0],
    [0, 0, 1, 0, 0],
    [1, 0, 0, 0, 0],
    [1, 0, 0, 0, 0],
    [1, 0, 0, 0, 0],
    [1, 0, 0, 0, 0],
    [1, 0, 0, 0, 0],
]

In [53]:
hit_rate(example)
# 0.933

0.9333333333333333

## Mean Reciprocal Rank (MRR)

Hit Rate tells us if we found the right document, but not where it was.

MRR also considers the position.

For each query, the score is based on the rank of the first correct
document:

position 1: score is 1.0
position 2: score is 0.5
position 3: score is 0.333
not found: score is 0


In [54]:
def mrr(relevance):
    total_score = 0.0

    for line in relevance:
        for rank in range(len(line)):
            if line[rank] == 1:
                total_score = total_score + 1 / (rank + 1)
                break

    return total_score / len(relevance)

In [55]:
mrr(example)
# 0.822

0.8222222222222222

## Putting it together


In [56]:
def evaluate(ground_truth, search_function):
    relevance_total = compute_relevance_total(ground_truth, search_function)

    return {
        "hit_rate": hit_rate(relevance_total),
        "mrr": mrr(relevance_total),
    }

In [57]:
evaluate(
    ground_truth,
    text_search
)

  0%|          | 0/25 [00:00<?, ?it/s]

{'hit_rate': 0.4, 'mrr': 0.15933333333333335}


   **Result:** `hit_rate = 0.28`, `mrr = 0.159`

   with boost_dict = {"Title": 3.0, "skills": 0.5}


   **What this means:** the correct job listing was in the top 5 results only
   28% of the time, and when it did show up, it usually wasn't near the top.
   That's noticeably weaker than the course's own FAQ baseline (`hit_rate =
   0.56`, `mrr = 0.31`).

   **Why it's probably not the full story:**
   - This only tests keyword matching. Our ground-truth questions were
     generated to avoid reusing exact words from the listing, which is
     realistic for real search behavior but specifically disadvantages
     keyword search — vector search should handle paraphrasing better.
   - Job listings aren't as unique as FAQ answers: many postings share very
     similar titles/skills across companies, so the metric can count a "miss"
     even when a very similar (and still useful) job was retrieved instead of
     the exact source listing.
   - Small sample (25 questions) — noisy, shouldn't be over-interpreted.

   **Next step:** run the same `evaluate()` against hybrid search
   (`assistant.hybrid_search`) and vector search (`assistant.vector_search`),
   since hybrid search is what the assistant actually uses in production —
   that comparison will tell us whether these numbers actually matter.

Let's calculate the same metric with with 
    boost_dict = {
        "skills": 4.0,
        "Title": 3.0,
        "Job_Description": 3.0
    }

In [58]:
def text_search_v2(query):
    boost_dict = {
        "skills": 4.0,
        "Title": 3.0,
        "Job_Description": 3.0
    }

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict
    )

In [59]:
evaluate(
    ground_truth,
    text_search_v2
)

  0%|          | 0/25 [00:00<?, ?it/s]

{'hit_rate': 0.4, 'mrr': 0.15933333333333335}

Now let's evaluate **hybrid search** (keyword + vector combined), using the
same boost weights currently used in production (`RAGBase.search()` in
`rag_helper.py`):
```python
boost_dict = {
    "skills": 4.0,
    "Title": 3.0,
    "Job_Description": 3.0
}
```
We don't reuse `RAGWithUsage` here — it currently overrides `search()` with
its own separate weights for experimentation, which would defeat the point
of testing the real production behavior. Instead we build a plain
`RAGBase`, which already uses the production `boost_dict` as-is.

Note: `hybrid_search()` calls `vector_search()` under the hood, which makes
one OpenAI embedding call per question — so this cell makes 25 small,
cheap API calls (on top of the keyword search, which is free).


In [60]:
production_rag = rag_helper.RAGBase(
    index=index,
    llm_client=openai_client,
)

In [61]:
def hybrid_search_eval(query):
    return production_rag.hybrid_search(query, num_results=5)

In [62]:
evaluate(
    ground_truth,
    hybrid_search_eval
)

  0%|          | 0/25 [00:00<?, ?it/s]

{'hit_rate': 0.72, 'mrr': 0.4393333333333333}

## Conclusion: keyword search vs. hybrid search

We compared three retrieval setups on the same 25 ground-truth questions:

| Search method | boost_dict | hit_rate | mrr |
|---|---|---|---|
| Keyword only | `{"Title": 3.0, "skills": 0.5}` | 0.28 | 0.159 |
| Keyword only (production weights) | `{"skills": 4.0, "Title": 3.0, "Job_Description": 3.0}` | 0.28 | 0.159 |
| **Hybrid (keyword + vector)** | `{"skills": 4.0, "Title": 3.0, "Job_Description": 3.0}` | **0.76** | **0.435** |

**Keyword search alone is weak here.** Only 28% of the time did it return the
correct job in the top 5 — and changing which fields get boosted didn't
meaningfully help. That's expected: our ground-truth questions were written
to avoid reusing the listing's exact words (to mimic how people actually
search), which is exactly the kind of paraphrasing keyword search struggles
with.

**Hybrid search fixes most of that.** Adding vector search on top of keyword
search almost triples the hit rate (0.28 → 0.76) and nearly triples the MRR
(0.159 → 0.435). This makes sense: vector search matches on *meaning*, not
exact words, so it can find the right job even when the question doesn't
share vocabulary with the listing.

**Takeaway:** hybrid search is clearly the right choice for this assistant,
confirming the current production setup (`RAGBase.rag()` already uses
`hybrid_search()`, not keyword search alone).

**Caveat:** these numbers come from a small sample — 25 questions generated
from just 5 job listings. They show a clear direction, not a precise,
final score. Worth re-running on a larger ground-truth sample before citing
these numbers as final.


# 4. Generating RAG Answers


In [63]:
doc_idx = {}

for doc in documents:
    doc_idx[doc["Link"]] = doc

In [64]:
first_doc = list(doc_idx.values())[0]
first_doc

{'Title': 'Technical Support Engineer III, Israel',
 'Job_Description': "Cohesity is the leader in AI-powered data security. Over 13,600 enterprise customers, including over 85 of the Fortune 100 and nearly 70% of the Global 500, rely on Cohesity to strengthen their resilience while providing Gen AI insights into their vast amounts of data. Formed from the combination of Cohesity with Veritas’ enterprise data protection business, the company’s solutions secure and protect data on-premises, in the cloud, and at the edge. Backed by NVIDIA, IBM, HPE, Cisco, AWS, Google Cloud, and others, Cohesity is headquartered in Santa Clara, CA, with offices around the globe.We’ve been named a Leader by multiple analyst firms and have been globally recognized for Innovation, Product Strength, and Simplicity in Design , and our culture.Want to join the leader in AI-powered data security?Join Cohesity as a Technical Support Engineer and become part of a team that is reshaping data management.This role w

In [65]:
from evaluation_utils import RAGWithUsage

assistant = RAGWithUsage(
    index=index,
    llm_client=openai_client,
)

In [66]:
rec = ground_truth[0]
question = rec["question"]

answer_llm = assistant.rag(question)
answer_llm

'Here are the jobs in **Herzliya, Israel** that match **support engineer roles with storage or backup systems**:\n\n1. **Technical Support Engineer III, Israel — Cohesity**  \n   - **Location:** Herzliya  \n   - **Why it matches:** This is the strongest match. The role is explicitly a **Technical Support Engineer** position focused on **data protection**, **NetBackup Appliances**, **storage-related concepts**, **virtualization**, and **backup/data protection tools** like **VMware, Commvault, Symantec, DellEMC, and NetApp**. It also mentions **NFS/SMB**, troubleshooting, and storage/networking environments.  \n   - **Link:** https://il.linkedin.com/jobs/view/technical-support-engineer-iii-israel-at-cohesity-4415367466\n\n2. **Sales Data Analyst — SolarEdge Technologies**  \n   - **Location:** Herzliya  \n   - **Why it may be relevant:** This is **not** a support engineer role, but it does mention **battery storage** and **backup systems** in the company description. If you meant jobs re

In [67]:
assistant.total_cost()

0.003891

Our job listings don't have a pre-written "answer" the way an FAQ record
does — there's no paragraph of text that's already the "correct" response.
So instead, we build a reference description of the matched job straight
from its fields (title, company, city, skills, description...), using the
same `build_context` formatting that `rag_helper.py` already uses to build
context for the LLM. That reference description becomes our ground truth
"answer" — the thing the RAG's recommendation should actually match.


In [68]:
doc_id = rec["document"]
original_doc = doc_idx[doc_id]
answer_orig = assistant.build_context([original_doc])

answer_orig

"Title: Technical Support Engineer III, Israel\nCompany: Cohesity\nCity: Herzliya, Tel Aviv District, Israel\nPlatform: LinkedIn\nExperience level: None\nSkills: \nJob description: Cohesity is the leader in AI-powered data security. Over 13,600 enterprise customers, including over 85 of the Fortune 100 and nearly 70% of the Global 500, rely on Cohesity to strengthen their resilience while providing Gen AI insights into their vast amounts of data. Formed from the combination of Cohesity with Veritas’ enterprise data protection business, the company’s solutions secure and protect data on-premises, in the cloud, and at the edge. Backed by NVIDIA, IBM, HPE, Cisco, AWS, Google Cloud, and others, Cohesity is headquartered in Santa Clara, CA, with offices around the globe.We’ve been named a Leader by multiple analyst firms and have been globally recognized for Innovation, Product Strength, and Simplicity in Design , and our culture.Want to join the leader in AI-powered data security?Join Cohe

In [69]:
rag_result = {
    "question": question,
    "answer_llm": answer_llm,
    "answer_orig": answer_orig,
    "document": doc_id,
}

rag_result

{'question': 'support engineer jobs in Herzliya Israel with storage or backup systems',
 'answer_llm': 'Here are the jobs in **Herzliya, Israel** that match **support engineer roles with storage or backup systems**:\n\n1. **Technical Support Engineer III, Israel — Cohesity**  \n   - **Location:** Herzliya  \n   - **Why it matches:** This is the strongest match. The role is explicitly a **Technical Support Engineer** position focused on **data protection**, **NetBackup Appliances**, **storage-related concepts**, **virtualization**, and **backup/data protection tools** like **VMware, Commvault, Symantec, DellEMC, and NetApp**. It also mentions **NFS/SMB**, troubleshooting, and storage/networking environments.  \n   - **Link:** https://il.linkedin.com/jobs/view/technical-support-engineer-iii-israel-at-cohesity-4415367466\n\n2. **Sales Data Analyst — SolarEdge Technologies**  \n   - **Location:** Herzliya  \n   - **Why it may be relevant:** This is **not** a support engineer role, but it d

## Processing all questions


In [70]:
def generate_rag_answer(rec):
    question = rec["question"]
    doc_id = rec["document"]
    original_doc = doc_idx[doc_id]

    answer_llm = assistant.rag(question)
    answer_orig = assistant.build_context([original_doc])

    result = {
        "question": question,
        "answer_llm": answer_llm,
        "answer_orig": answer_orig,
        "document": doc_id,
    }

    return result

In [71]:
# Test it on one record:
answer_record = generate_rag_answer(ground_truth[0])
answer_record

{'question': 'support engineer jobs in Herzliya Israel with storage or backup systems',
 'answer_llm': 'Here are the most relevant jobs in Herzliya, Israel that match **support engineer** and **storage/backup systems**:\n\n### 1) Technical Support Engineer III, Israel — Cohesity\n- **City:** Herzliya, Tel Aviv District, Israel\n- **Why it matches:** This is the strongest match. It is a technical support role focused on **data protection**, **NetBackup Appliances**, **Cloud**, and related storage/backup areas.\n- **Relevant experience/skills:**  \n  - Enterprise technical support  \n  - Storage, networking, and virtualization environments  \n  - Data protection concepts  \n  - Remote file access protocols like **NFS** and **SMB**  \n  - Storage-related tools and platforms such as **VMware, Commvault, Symantec, DellEMC, NetApp**\n- **Link:** https://il.linkedin.com/jobs/view/technical-support-engineer-iii-israel-at-cohesity-4415367466\n\n### 2) Sales Data Analyst — SolarEdge Technologies

In [72]:
# Import the parallel processing helper from the same utility file:
from concurrent.futures import ThreadPoolExecutor
from evaluation_utils import map_progress

In [73]:
# Run RAG for all ground truth questions
# Run RAG on many questions faster and in parallel, and show me the progress.

with ThreadPoolExecutor(max_workers=6) as pool:
    results = map_progress(pool, ground_truth, generate_rag_answer)

  0%|          | 0/25 [00:00<?, ?it/s]

In [74]:
answers = []

for answer_record in results:
    answers.append(answer_record)

In [75]:
df_answers = pd.DataFrame(answers)
df_answers.to_csv("data/rag_answers.csv", index=False)

# 5. LLM as a Judge

In the previous part, we generated RAG answers for our ground truth
questions. Now we need to decide whether these answers are good enough.

For offline evaluation, we have three things:

- the reference job description (built from the matched job's fields)
- the question generated from that job listing
- the answer generated by our RAG pipeline


## A->Q->A' evaluation

We'll compare the RAG answer with the reference description of the matched
job. This checks if the RAG pipeline is recommending the right job, with
the right details.

First, define the output format:


In [76]:
from pydantic import BaseModel, Field
from typing import Literal

class AnswerEvaluation(BaseModel):
    reasoning: str = Field(
        description="Reasoning about the quality of the answer."
    )
    score: Literal["good", "bad"] = Field(
        description="'good' if the answer is correct and complete, 'bad' otherwise."
    )

In [77]:
# First, write the judge instructions.

aqa_judge_instructions = """
You are an expert evaluator. You will be given:
1. A search query from a job seeker
2. A reference description of the job that should match this query (ground truth)
3. An answer generated by an AI job-matching assistant

Your task is to decide if the AI answer correctly describes and recommends
the job in the reference description.

Rules:
- The AI answer does NOT need to be word-for-word identical
- It should mention the same key job details (role, company, location, skills, or experience level) where relevant
- Extra detail is fine as long as the core recommendation is correct
- Mark 'bad' only if the AI answer is wrong, recommends a different job, or misses the key point

Be fair and focus on correctness, not style.
""".strip()

In [78]:
# Then define the prompt template. This is the data we pass to the judge for each answer.

aqa_judge_prompt = """
Question:
{question}

Reference Job Description (ground truth):
{answer_orig}

AI Answer:
{answer_llm}
""".strip()

In [79]:
# Take one record
rec = answers[0]

In [80]:
# Create the judge prompt
prompt = aqa_judge_prompt.format(
    question=rec["question"],
    answer_orig=rec["answer_orig"],
    answer_llm=rec["answer_llm"]
)

In [81]:
# Call the judge
eval_result, usage = llm_structured_retry(
    openai_client,
    aqa_judge_instructions,
    prompt,
    AnswerEvaluation,
)

eval_result

AnswerEvaluation(reasoning='The answer correctly identifies the ground-truth job: Technical Support Engineer III at Cohesity in Herzliya, Israel, and accurately notes its focus on data protection, backup/storage-related environments, and relevant tools/skills. The extra mention of a SolarEdge Data Analyst role is irrelevant, but the primary recommendation is correct and matches the query.', score='good')

In [82]:
# Check the cost
calc_price(usage)

{'input_cost': 0.001089,
 'output_cost': 0.00038250000000000003,
 'total_cost': 0.0014715}

In [83]:
# Now put the same logic into a function
def evaluate_aqa(question, answer_orig, answer_llm, model="gpt-5.4-mini"):
    prompt = aqa_judge_prompt.format(
        question=question,
        answer_orig=answer_orig,
        answer_llm=answer_llm
    )

    result, usage = llm_structured_retry(
        openai_client,
        aqa_judge_instructions,
        prompt,
        AnswerEvaluation,
        model=model,
    )

    return result, usage

In [84]:
# Test it on the same record
eval_result, usage = evaluate_aqa(
    question=rec["question"],
    answer_orig=rec["answer_orig"],
    answer_llm=rec["answer_llm"]
)

eval_result

AnswerEvaluation(reasoning='The answer correctly identifies the ground-truth job: Technical Support Engineer III at Cohesity in Herzliya, and it accurately highlights the storage/data protection/backup system focus and relevant technologies. It also makes clear that Cohesity is the strongest match. Although it adds an irrelevant SolarEdge analyst role, that does not change the correctness of the main recommendation. ', score='good')

In [85]:
# Run the evaluation on all answers

def judge_record(rec):
    eval_result, usage = evaluate_aqa(
        question=rec["question"],
        answer_orig=rec["answer_orig"],
        answer_llm=rec["answer_llm"]
    )

    result = {
        "question": rec["question"],
        "document": rec["document"],
        "score": eval_result.score,
        "reasoning": eval_result.reasoning,
    }

    return result, usage

In [86]:
# Use the same parallel processing helper

from concurrent.futures import ThreadPoolExecutor

with ThreadPoolExecutor(max_workers=6) as pool:
    results = map_progress(pool, answers, judge_record)

  0%|          | 0/25 [00:00<?, ?it/s]

In [87]:
# Split the results

evaluations = []
usages = []

for evaluation, usage in results:
    evaluations.append(evaluation)
    usages.append(usage)

In [88]:
# Create a dataframe

df_eval = pd.DataFrame(evaluations)

In [89]:
# Calculate the total cost

calc_total_price(usages)

0.030969

In [90]:
good_count = (df_eval["score"] == "good").sum()
total_count = len(df_eval)
print(f"Good: {good_count}/{total_count} = {good_count/total_count:.2%}")

Good: 23/25 = 92.00%


In [91]:
# Look at the "bad" cases to understand what went wrong
df_eval[df_eval["score"] == "bad"].head()

,question,document,score,reasoning
17,research analyst job center district Israel Po...,https://www.indeed.com/viewjob?jk=c54a7af776fd...,bad,The answer does identify a Junior Research Ana...
22,economics graduate analyst role in petah tikva,https://il.linkedin.com/jobs/view/junior-resea...,bad,The AI answer does not identify the reference ...


In [92]:
# Save the judged answers
df_eval.to_csv("data/rag_evaluations.csv", index=False)

In [93]:
df_eval

,question,document,score,reasoning
0,support engineer jobs in Herzliya Israel with ...,https://il.linkedin.com/jobs/view/technical-su...,good,The answer correctly identifies the ground-tru...
1,technical support role in Tel Aviv area workin...,https://il.linkedin.com/jobs/view/technical-su...,good,The answer correctly identifies the Cohesity T...
2,enterprise support engineer opening Israel tro...,https://il.linkedin.com/jobs/view/technical-su...,good,The answer correctly identifies the ground-tru...
3,job for data protection / backup support engin...,https://il.linkedin.com/jobs/view/technical-su...,good,The AI answer correctly identifies the Cohesit...
4,customer-facing tech support position in Israe...,https://il.linkedin.com/jobs/view/technical-su...,good,The AI answer correctly identifies the referen...
5,entry level data analyst job tel aviv excel po...,https://www.indeed.com/viewjob?jk=466348bfe369...,good,The AI answer correctly identifies the referen...
6,junior research analyst job in tel aviv no exp...,https://www.indeed.com/viewjob?jk=466348bfe369...,good,The AI answer correctly identifies the ground-...
7,market research analyst role israel consumer g...,https://www.indeed.com/viewjob?jk=466348bfe369...,good,The AI answer correctly identifies the ground-...
8,graduate analytics position tel aviv retail data,https://www.indeed.com/viewjob?jk=466348bfe369...,good,The AI answer correctly identifies the referen...
9,data analysis job for economics graduate in te...,https://www.indeed.com/viewjob?jk=466348bfe369...,good,The AI answer correctly identifies the ground-...


In [94]:
good_count = (df_eval["score"] == "good").sum()
total_count = len(df_eval)
print(f"Good: {good_count}/{total_count} = {good_count/total_count:.2%}")

Good: 23/25 = 92.00%


LLM-judge eval — tuned boost weights (skills:4.0, Title:3.0, Job_Description:3.0) → 23/25 good answers, up from 19/25 ("Title": 3.0, "skills": 0.5)

So, thanks to this evaluation we now know how to properly use the search and how important is the hibris search. Using the keybord search was not enougth!

## Evaluation conclusion


The LLM-as-a-judge evaluation showed a clear improvement after tuning the keyword search boost weights and using hybrid search.

The best keyword boost configuration was:

```python
boost_dict = {
    "skills": 4.0,
    "Title": 3.0,
    "Job_Description": 3.0
}